In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['NEO4J_URI'] = "neo4j://44.202.197.250:7687"
os.environ['NEO4J_USERNAME'] = "neo4j"
os.environ['NEO4J_PASSWORD'] = "crime-laws-student"

In [2]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]
    db_outputs: list

graph_builder = StateGraph(State)

In [3]:
from neo4j import GraphDatabase, basic_auth

neo4j_uri = os.environ['NEO4J_URI']
neo4j_username = os.environ['NEO4J_USERNAME']
neo4j_password = os.environ['NEO4J_PASSWORD']

driver = GraphDatabase.driver(
    neo4j_uri,
    auth=basic_auth(neo4j_username, neo4j_password)
)

In [4]:
from neo4j.time import Date

def get_node_datatype(value):
    '''
    입력된 노드 Value의 데이터 타입을 반환하는 함수
    '''
    if isinstance(value, str):
        return 'STRING'
    elif isinstance(value, int):
        return 'INTEGER'
    elif isinstance(value, float):
        return 'FLOAT'
    elif isinstance(value, bool):
        return 'BOOLEAN'
    elif isinstance(value, list):
        return f'LIST[{get_node_datatype(value[0])}]' if value else "LIST"
    elif isinstance(value, Date):
        return 'DATE'
    else:
        return 'UNKNOWN'

In [5]:
def get_schema_dict():
    '''
    Graph DB의 정보를 받아 노드 및 관계의 프로퍼티를 추출하고 스키마 딕셔너리를 반환하는 함수
    '''
    with driver.session() as session:
        node_query = '''
        MATCH (n)
        WITH DISTINCT labels(n) AS node_labels, keys(n) AS property_keys, n
        UNWIND node_labels AS label
        UNWIND property_keys AS key
        RETURN label, key, n[key] AS sample_value
        '''
        nodes = session.run(node_query)

        rel_query = '''
        MATCH ()-[r]->()
        WITH DISTINCT type(r) AS rel_type, keys(r) AS property_keys, r
        UNWIND property_keys AS key
        RETURN rel_type, key, r[key] AS sample_value
        '''
        relationships = session.run(rel_query)
        
        rel_direction_query = '''
        MATCH (a)-[r]->(b)
        RETURN DISTINCT labels(a) AS start_label, type(r) AS rel_type, labels(b) AS end_label
        ORDER BY start_label, rel_type, end_label
        '''
        rel_directions = session.run(rel_direction_query)

        schema = {'nodes': {}, 'relationships': {}, 'relations': []}

        for record in nodes:
            label = record['label']
            key = record['key']
            sample_value = record['sample_value']
            inferred_type = get_node_datatype(sample_value)
            if label not in schema['nodes']:
                schema['nodes'][label] = {}
            schema['nodes'][label][key] = inferred_type
        
        for record in relationships:
            rel_type = record['rel_type']
            key = record['key']
            sample_value = record['sample_value']
            inferred_type = get_node_datatype(sample_value)
            if rel_type not in schema['relationships']:
                schema['relationships'][rel_type] = {}
            schema['relationships'][rel_type][key] = inferred_type
        
        for record in rel_directions:
            start_label = record['start_label'][0]
            rel_type = record['rel_type']
            end_label = record['end_label'][0]
            schema['relations'].append(f'(:{start_label})-[:{rel_type}]->(:{end_label})')
        
        return schema

def get_schema_str(schema):
    result = []

    result.append('Node properties:')
    for label, properties in schema['nodes'].items():
        props = ', '.join(f'{k}: {v}' for k, v in properties.items())
        result.append(f'{label} {{{props}}}')
    
    result.append('Relationship properties:')
    for rel_type, properties in schema['relationships'].items():
        props = ', '.join(f'{k}: {v}' for k, v in properties.items())
        result.append(f'{rel_type} {{{props}}}')

    result.append('The relationships:')
    for relation in schema['relations']:
        result.append(relation)
    
    return '\n'.join(result)

In [6]:
schema = get_schema_str(get_schema_dict())

In [7]:
print(schema)

Node properties:
Movie {url: STRING, runtime: INTEGER, revenue: INTEGER, budget: INTEGER, imdbRating: FLOAT, released: STRING, countries: LIST[STRING], languages: LIST[STRING], plot: STRING, imdbVotes: INTEGER, imdbId: STRING, year: INTEGER, poster: STRING, movieId: STRING, tmdbId: STRING, title: STRING}
Genre {name: STRING}
User {userId: STRING, name: STRING}
Actor {bornIn: STRING, born: DATE, died: DATE, tmdbId: STRING, imdbId: STRING, name: STRING, url: STRING, bio: STRING, poster: STRING}
Person {bornIn: STRING, born: DATE, died: DATE, tmdbId: STRING, imdbId: STRING, name: STRING, url: STRING, bio: STRING, poster: STRING}
Director {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}
Relationship properties:
RATED {rating: FLOAT, timestamp: INTEGER}
ACTED_IN {role: STRING}
DIRECTED {role: STRING}
The relationships:
(:Actor)-[:ACTED_IN]->(:Movie)
(:Actor)-[:DIRECTED]->(:Movie)
(:Actor)-[:ACTED_IN]->(:Movie)
(

In [8]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')

In [9]:
fewshot_examples = [
    "USER INPUT: 'Toy Story'에 어떤 배우들이 출연하나요?' QUERY: MATCH (a:Actor)-[:ACTED_IN]->(m:Movie) WHERE m.title = 'Toy Story' RETURN a.name",
    "USER INPUT: 'Toy Story의 평균 평점은 몇점인가요?' QUERY: MATCH (u:User)-[r:Rated]->(m:Movie) WHERE m.title = 'Toy Story' RETURN AVG(r.rating)",
]

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage

GENERATE_SYSTEM_TEMPLATE = '''Given an input question, convert it to a Cypher query. No pre-amble.
Do not wrap the response in any backticks or anything else. Respond with a Cypher statement only!'''

GENERATE_USER_TEMPLATE = '''You are a Neo4j expert. Given an input question, create a syntactically correct Cypher query to run.
Do not wrap the response in any backticks or anything else. Respond with a Cypher statement only!
Here is the schema information
{schema}

Below are a number of examples of questions and their corresponding Cypher queries.

{fewshot_examples}

User input: {question}
Cypher query:'''

def generate_cypher(state: State):
    generate_cypher_msgs = [
        ('system', GENERATE_SYSTEM_TEMPLATE),
        ('user', GENERATE_USER_TEMPLATE)
    ]
    text2cypher_prompt = ChatPromptTemplate.from_messages(generate_cypher_msgs)

    response = llm.invoke(
        text2cypher_prompt.format_messages(
            question=state['messages'], schema=schema, fewshot_examples=fewshot_examples
        )
    )
    outputs = []
    outputs.append(
        AIMessage(
            content=response.content,
        )
    )

    return {'messages': outputs}

graph_builder.add_node('generate_cypher', generate_cypher)

In [11]:
import json

class ExecuteCypherNode:
    def __init__(self) -> None:
        self.driver = driver

    def __call__(self, inputs: dict):
        print('###### EXECUTE CYPHER ######')
        if messages := inputs.get('messages', []):
            message = messages[-1].content
        else:
            raise ValueError('No message found in input')
        outputs = []

        print('실행 쿼리문', message)

        try:
            with driver.session(database='neo4j') as session:
                database_output = session.read_transaction(
                    lambda tx: tx.run(message).data()
                )
        except Exception as e:
            database_output = str(e)
        
        outputs.append(
            database_output
        )
        return {'db_outputs': outputs}
    
execute_cypher_node = ExecuteCypherNode()
graph_builder.add_node('execute_cypher', execute_cypher_node)

In [12]:
def route_correction(
        state: State,
):
    if db_outputs := state.get('db_outputs', []):
        db_result = db_outputs[-1]
    else:
        raise ValueError(f'No DB result found')
    
    print('###### ROUTE QUERY CORRECTION ######')
    print('DB 조회 결과', db_result)
    if type(db_result) == list and len(db_result) > 0:
        print('!정상 조회 완료!')
        return 'answer'
    
    print('!정상 조회 실패!')
    return 'correct_cypher'

graph_builder.add_conditional_edges(
    'execute_cypher',
    route_correction,
    {'correct_cypher': 'correct_cypher', 'answer': 'answer'}
)